# veritract — verified extraction demo

**Verified extraction: truth-anchored LLM structured output.**

LLMs extract plausible-looking JSON — but how do you know which fields to trust?
veritract answers that question with a grounding layer that traces every extracted
value back to a character span in the source, and quarantines anything it can't verify.

```
LLM extraction  →  fuzzy grounding  →  LLM re-verify  →  typed provenance + quarantine
```

**Provenance types:**
| Type | Meaning |
|---|---|
| `direct` | Exact substring — value appears verbatim in source |
| `paraphrased` | Fuzzy match above threshold |
| `inferred` | LLM confirmed; fuzzy failed (abbreviation, unit conversion) |
| quarantined | Neither could verify — surfaced for human review |

## Setup

**Running this notebook:**
1. Make sure [Ollama](https://ollama.com) is running: `ollama serve`
2. Pull the model: `ollama pull gemma4:e4b`
3. Select the **veritract (venv)** kernel in Jupyter (Kernel menu → Change Kernel)

The venv has all dependencies pre-installed including `docling` for PDF and figure support.
If running in a fresh environment, install from source (not yet on PyPI):
```
git clone https://github.com/LanGuo/veritract.git && cd veritract
pip install -e '.[pdf]'
```

In [ ]:
import textwrap
from veritract import extract, extract_raw, ground, LLMClient, MockLLM, load_images_b64

# Real Ollama model — runs fully locally, no API key needed
llm = LLMClient(model="gemma4:e4b", temperature=0.0, seed=42)

---
## Part 1 — Define your schema

The schema is standard [JSON Schema](https://json-schema.org/). Each property is a field
you want extracted. All fields in `required` will always appear in the output —
GBNF-constrained decoding forces the model to emit every field; unconfident
extractions surface as quarantined rather than silently missing.

In [ ]:
# Clinical trial abstract — our source document
ABSTRACT = """
A double-blind randomised controlled trial evaluated the efficacy of atorvastatin
40 mg daily versus placebo in 486 patients with moderate hypercholesterolaemia.
Participants were followed for 52 weeks. The primary endpoint was reduction in
LDL-cholesterol from baseline, which decreased by 43% in the atorvastatin group
compared to 4% in the placebo group (p < 0.0001). Treatment was well tolerated;
elevated liver enzymes were recorded in 3 patients (0.6%).
""".strip()

# Define what you want to extract — plain JSON Schema
SCHEMA = {
    "type": "object",
    "properties": {
        "drug":            {"type": "string"},
        "sample_size":     {"type": "string"},
        "primary_outcome": {"type": "string"},
    },
    "required": ["drug", "sample_size", "primary_outcome"],
}

print("Source text:")
print(textwrap.fill(ABSTRACT, 80))

---
## Part 2 — Write your prompt

The prompt describes **what each field means** and **how to extract it**.
Key rules to include:
- Copy verbatim phrases — don't paraphrase or synthesise
- Return an empty string if a field isn't present (not "N/A" or "not found")
- Don't use prior knowledge — only extract from the provided text

If you omit the prompt, veritract generates a default one from the schema field names.

In [ ]:
# A good prompt: field descriptions + verbatim-copy rule
PROMPT = """You are extracting facts from a clinical trial abstract.
Rules:
- Copy the exact verbatim phrase from the text. Do not paraphrase or use prior knowledge.
- If a field is not present in this excerpt, return an empty string.

Fields:
* drug: Drug name and dose as stated (e.g. "atorvastatin 40 mg daily").
* sample_size: Number of patients enrolled as stated.
* primary_outcome: Primary endpoint or outcome measure as stated."""

# One-call extraction + grounding — text is the document to extract from
result = extract(ABSTRACT, SCHEMA, llm, prompt=PROMPT, mode="full")

print("Extracted fields:")
for field, gf in result.extracted.items():
    span = gf["span"]
    ptype = span["provenance_type"] if span else "no-span"
    print(f"  [{ptype:>12}]  {field}: {gf['value']!r}")

if result.quarantined:
    print("\nQuarantined:")
    for q in result.quarantined:
        print(f"  {q['field_name']}: {q['value']!r} — {q['reason']}")

In [ ]:
# Every grounded field has a character span — locate it in the source
def highlight(text, field_name, result):
    gf = result.extracted.get(field_name)
    if not gf or not gf["span"]:
        return f"(no span for {field_name})"
    s, e = gf["span"]["char_start"], gf["span"]["char_end"]
    return text[:s] + ">>" + text[s:e] + "<<" + text[e:]

for field in result.extracted:
    print(f"{field}:")
    print(textwrap.fill(highlight(ABSTRACT, field, result), 80))
    print()

---
## Part 3 — Optional: few-shot examples

Pass `examples` to show the model what good extractions look like.
Each example is a dict with `"text"` and `"fields"`. Examples are injected
into the prompt automatically — useful when field boundaries are ambiguous
or the model misinterprets a field.

In [ ]:
EXAMPLES = [
    {
        "text": (
            "A randomised trial of metformin 500mg twice daily enrolled 120 patients "
            "with type 2 diabetes. The primary outcome was HbA1c reduction at 6 months."
        ),
        "fields": {
            "drug":            "metformin 500mg twice daily",
            "sample_size":     "120 patients",
            "primary_outcome": "HbA1c reduction at 6 months",
        },
    },
]

result_with_examples = extract(
    ABSTRACT, SCHEMA, llm,
    prompt=PROMPT,
    examples=EXAMPLES,
    mode="full",
)

print("With examples:")
for field, gf in result_with_examples.extracted.items():
    ptype = gf["span"]["provenance_type"] if gf["span"] else "no-span"
    print(f"  [{ptype:>12}]  {field}: {gf['value']!r}")

---
## Part 4 — Optional: prompt optimization

`optimize_prompt` runs iterative refinement: extract → score by grounding rate →
ask the LLM to suggest improvements → repeat. Returns the highest-scoring prompt.

Pass `ground_truth` labels for supervised scoring (accuracy instead of grounding rate).
Benchmarks show **10–20 percentage point accuracy gains** on clinical NLP tasks after 3–5 iterations.

In [ ]:
from veritract import optimize_prompt

# A small set of examples to optimise over (use more in practice)
TRAIN_EXAMPLES = [
    {"text": ABSTRACT},
    {
        "text": (
            "We conducted a placebo-controlled study of rosuvastatin 20mg in 312 patients "
            "with dyslipidaemia over 24 weeks. The primary endpoint was LDL-C change from baseline."
        ),
    },
]

# Unsupervised: score by grounding rate (no labels needed)
best_prompt = optimize_prompt(TRAIN_EXAMPLES, SCHEMA, llm, n_iter=2)

print("Optimised prompt (first 300 chars):")
print(best_prompt[:300], "...")

# Use the optimised prompt for extraction
result_opt = extract(ABSTRACT, SCHEMA, llm, prompt=best_prompt)
print("\nWith optimised prompt:")
for field, gf in result_opt.extracted.items():
    ptype = gf["span"]["provenance_type"] if gf["span"] else "no-span"
    print(f"  [{ptype:>12}]  {field}: {gf['value']!r}")

---
## Part 5 — The grounding layer: catching hallucinations

Standard extraction pipelines return valid-looking JSON with no way to tell
a verbatim extraction from a confabulation. Here's what that looks like —
we inject a hallucinating model and run with **no grounding**:

In [ ]:
# MockLLM simulates a hallucinating model — wrong drug, wrong count, wrong outcome
hallucinating_llm = MockLLM()
hallucinating_llm.register(ABSTRACT[:30], {
    "drug":            "rosuvastatin 20 mg",   # wrong
    "sample_size":     "250 patients",          # wrong
    "primary_outcome": "cardiovascular events", # wrong
})

raw = extract_raw(ABSTRACT, SCHEMA, hallucinating_llm)

print("Raw LLM output (no verification):")
for field, value in raw.fields.items():
    print(f"  {field}: {value!r}")
print("\nLooks fine — valid JSON, plausible values. But none appear in the source.")

In [ ]:
# Ground those raw results — all three hallucinations caught
result_h = ground(raw, llm, mode="fuzzy")

print(f"Grounded: {len(result_h.extracted)}  |  Quarantined: {len(result_h.quarantined)}")
print()
for q in result_h.quarantined:
    print(f"  QUARANTINED  {q['field_name']}: {q['value']!r}")
    print(f"               {q['reason']}")

---
## Part 6 — Two-stage pipeline: pay for LLM inference once

`extract_raw` and `ground` are independently callable. Extract once,
apply different grounding strategies — or save raw outputs for offline analysis.

In [ ]:
# Stage 1: LLM call — returns RawExtractionResult
raw = extract_raw(ABSTRACT, SCHEMA, llm, prompt=PROMPT)
print("Stage 1 — raw LLM output:")
for f, v in raw.fields.items():
    print(f"  {f}: {v!r}")

print()

# Stage 2: apply different grounding strategies to the same raw output
for mode_name in ("fuzzy", "full", "no-grounding"):
    res = ground(raw, llm, mode=mode_name)
    n_grounded = sum(1 for gf in res.extracted.values() if gf["span"])
    print(f"  mode={mode_name!r:15}  grounded={n_grounded}  quarantined={len(res.quarantined)}")

---
## Part 7 — PDF extraction (docling + chunking)

`extract_pdf` converts PDF→markdown via docling (handles text, tables, scanned pages),
chunks the markdown, extracts from each chunk, merges results (preferring compact,
high-frequency values), and grounds against the full document.

In [ ]:
from veritract import extract_pdf

TECH_SCHEMA = {
    "type": "object",
    "properties": {
        "architecture_type":       {"type": "string"},
        "total_parameters":        {"type": "string"},
        "context_length":          {"type": "string"},
        "pretraining_token_count": {"type": "string"},
        "alignment_and_rl_method": {"type": "string"},
    },
    "required": [
        "architecture_type", "total_parameters", "context_length",
        "pretraining_token_count", "alignment_and_rl_method",
    ],
}

PDF_PROMPT = """You are extracting facts from an LLM technical report.
Rules:
- Copy the exact verbatim phrase from the text. Do not paraphrase or use prior knowledge.
- If a field is not present in this specific excerpt, return an empty string.

Fields:
* architecture_type: Model architecture family and variant (dense, MoE, attention type).
* total_parameters: Total parameter count as stated.
* context_length: Maximum context window as stated.
* pretraining_token_count: Total pretraining tokens as stated.
* alignment_and_rl_method: Post-SFT alignment method (RLHF, DPO, GRPO, BOND, etc.)."""

# Download: https://arxiv.org/pdf/2503.19786  (Gemma 3 technical report)
PDF_PATH = "/tmp/llm_reports/gemma3.pdf"

print("Extracting from Gemma 3 technical report...")
result_pdf = extract_pdf(
    PDF_PATH, TECH_SCHEMA, llm,
    chunk_size=8000, chunk_overlap=500,
    mode="fuzzy", prompt=PDF_PROMPT,
)

print("\nGrounded fields:")
for field, gf in result_pdf.extracted.items():
    if gf["value"]:
        ptype = gf["span"]["provenance_type"] if gf["span"] else "no-span"
        print(f"  [{ptype:>12}]  {field}: {gf['value'][:90]}")

real_q = [q for q in result_pdf.quarantined if q["value"]]
if real_q:
    print("\nQuarantined:")
    for q in real_q:
        print(f"  {q['field_name']}: {q['value'][:60]!r}")

---
## Part 8 — Multimodal: extracting from PDF figures

veritract supports image input for multimodal models. Here we extract structured
data directly from a figure in the Gemma 3 technical report.

Key design: grounding runs against the **full document text** — so values visible
in a chart but discussed anywhere in the paper body can be verified and get a span.
Values only readable from the image (e.g. chart type, axis tick values) are quarantined
— an honest signal that those fields came purely from vision.

In [ ]:
import base64, io
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat

FIGURE_SCHEMA = {
    "type": "object",
    "properties": {
        "figure_type":     {"type": "string"},
        "models_compared": {"type": "string"},
        "metric":          {"type": "string"},
        "best_result":     {"type": "string"},
        "key_finding":     {"type": "string"},
    },
    "required": ["figure_type", "models_compared", "metric", "best_result", "key_finding"],
}

FIGURE_PROMPT = """You are reading a figure from an ML research paper.
Rules:
- Extract only what is directly visible in the figure or stated in the caption.
- Copy exact text labels as they appear. Do not use prior knowledge.
- If a field is not visible or stated, return an empty string.

Fields:
* figure_type: Type of visual — bar chart, line plot, architecture diagram, table, etc.
* models_compared: Model names shown (comma-separated as labelled in the figure).
* metric: What is being measured (exact axis label or caption description).
* best_result: Highest or best numerical result visible, with model name.
* key_finding: Main takeaway stated in the caption or shown in the figure."""

# Load PDF with image extraction enabled
opts = PdfPipelineOptions()
opts.generate_picture_images = True
opts.images_scale = 2.0
conv = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=opts)}
)
doc = conv.convert(PDF_PATH)
full_text = doc.document.export_to_markdown()

# Pick Figure 2 — performance summary chart
qualifying = [
    (p.get_image(doc.document), p.caption_text(doc.document) or "")
    for p in doc.document.pictures
    if p.get_image(doc.document) and p.get_image(doc.document).width >= 200
]
img, caption = qualifying[1]
print(f"Figure: {img.width}×{img.height}px")
print(f"Caption: {caption[:100]}")

# Encode image as base64
buf = io.BytesIO()
img.save(buf, format="PNG")
b64 = base64.b64encode(buf.getvalue()).decode()

# Extract: prompt sees caption + image; grounding searches the full document
raw_fig = extract_raw(
    caption, FIGURE_SCHEMA, llm,
    prompt=FIGURE_PROMPT,
    images=[b64],          # pass image alongside text
    source_type="figure",
    doc_id="gemma3.pdf",
)
raw_fig.source_text = full_text   # ground against full doc, not just caption
result_fig = ground(raw_fig, llm=None, mode="fuzzy")

print("\nExtracted fields:")
for field, gf in result_fig.extracted.items():
    if gf["value"]:
        ptype = gf["span"]["provenance_type"] if gf["span"] else "no-span"
        print(f"  [{ptype:>12}]  {field}: {gf['value'][:90]}")

if result_fig.quarantined:
    print("\nQuarantined (vision-only — not found in document text):")
    for q in result_fig.quarantined:
        if q["value"]:
            print(f"  {q['field_name']}: {q['value'][:70]!r}")

---
## Benchmark — vs LangExtract on EBM-NLP 2.0

Evaluated on **155 expert-annotated clinical trial abstracts** (fields: `drug`, `sample_size`, `outcome`).
Both systems: `gemma4:e4b` via Ollama, 3 runs × 155 samples, temp=0.3, optimised prompt.

| | veritract | LangExtract |
|---|---|---|
| **Field accuracy** | **87.7% ± 0.3%** | 84.7% ± 1.9% |
| Extraction latency | 28.9s | **5.5s** |
| Verification cost | +5.4s | +0.2s |
| Grounded (has span) | **90.3%** | 85.7% |
| Quarantined | 9.7% | 0%* |
| Missing (never extracted) | **0%** | ~14.3% |
| Quarantine precision | **100%** | — |
| Quarantine recall | 79% | — |

*LangExtract skips fields it can't answer; veritract always emits every schema field.

**Quarantine precision 100%:** every quarantined field was genuinely wrong.
**Quarantine recall 79%:** the other 21% of wrong fields passed grounding — real phrases from the source but for the wrong field ("grounded hallucinations").

---
## Summary

| | veritract | Plain JSON / Instructor |
|---|---|---|
| Extracts structured JSON | ✓ | ✓ |
| Constrained decoding (no malformed JSON) | ✓ GBNF | ✓ (provider-dependent) |
| Every field traced to source span | ✓ | ✗ |
| Hallucinations caught and quarantined | ✓ | ✗ |
| Trust signal per field (not per response) | ✓ | ✗ |
| Fully local / air-gapped | ✓ Ollama | ✗ API required |
| PDF extraction with table support | ✓ docling | ✗ |
| Multimodal (image + figure) extraction | ✓ image-first preamble | provider-dependent |

**GitHub:** https://github.com/LanGuo/veritract
**Install (from source):** `git clone https://github.com/LanGuo/veritract.git && pip install -e '.[pdf]'`
*(PyPI release coming soon)*